<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/llama/llama_cpp_t4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# llama.cpp on a Colab T4

Build [llama.cpp](https://github.com/ggml-org/llama.cpp) with CUDA, download a GGUF model, and chat with it.

Runtime → Change runtime type → **T4 GPU**. The first build takes several minutes.

## 1. Detect GPU, CPU, and RAM

In [ ]:
import os, subprocess

# GPU summary
!nvidia-smi

# CPU + RAM every run
def _host():
    model, threads, sockets, core_ids = "unknown CPU", 0, set(), set()
    cur_phys = cur_core = None
    try:
        text = open("/proc/cpuinfo", encoding="utf-8", errors="replace").read()
    except OSError:
        text = ""
    for line in text.splitlines():
        if not line.strip():
            if cur_phys is not None and cur_core is not None:
                core_ids.add((cur_phys, cur_core))
            cur_phys = cur_core = None
            continue
        if ":" not in line:
            continue
        k, v = [x.strip() for x in line.split(":", 1)]
        if k == "model name" and model == "unknown CPU":
            model = v
        elif k == "processor":
            threads += 1
        elif k == "physical id":
            cur_phys = v; sockets.add(v)
        elif k == "core id":
            cur_core = v
    if cur_phys is not None and cur_core is not None:
        core_ids.add((cur_phys, cur_core))
    threads = threads or (os.cpu_count() or 1)
    cores = len(core_ids) or threads
    mem = {}
    try:
        for line in open("/proc/meminfo", encoding="utf-8", errors="replace"):
            if ":" not in line: continue
            k, rest = line.split(":", 1)
            parts = rest.split()
            if parts:
                try: mem[k] = int(parts[0])
                except ValueError: pass
    except OSError:
        pass
    total_kb = mem.get("MemTotal", 0)
    avail_kb = mem.get("MemAvailable", mem.get("MemFree", 0))
    to_gb = lambda kb: kb / 1_000_000
    print(f"CPU: {model} — {cores} cores / {threads} threads" + (f" ({len(sockets)} sockets)" if len(sockets) > 1 else ""))
    print(f"RAM: {to_gb(total_kb):.1f} GB total, {to_gb(avail_kb):.1f} GB available ({to_gb(total_kb-avail_kb):.1f} GB used)")
    try:
        u = os.statvfs("/content")
        print(f"Disk (/content): {u.f_bavail*u.f_frsize/1e9:.1f} GB free / {u.f_blocks*u.f_frsize/1e9:.1f} GB total")
    except OSError:
        pass
    return model, cores, threads, to_gb(total_kb), to_gb(avail_kb)

CPU_MODEL, CPU_CORES, CPU_THREADS, RAM_GB, RAM_AVAIL_GB = _host()


## 2. Config

In [ ]:
# Qwen3.8-27B, Unsloth dynamic Q4_K_M (~17.1 GB). Single-file quant.
REPO_ID   = "unsloth/Qwen3.8-27B-GGUF"
FILENAME  = "Qwen3.8-27B-UD-Q4_K_M.gguf"
# Smaller alternatives if this is too slow or runs out of memory:
#   FILENAME = "Qwen3.8-27B-UD-IQ4_XS.gguf"      (~15.7 GB, almost fits the T4 entirely)
#   REPO_ID = "bartowski/Qwen3.8-27B-GGUF"; FILENAME = "Qwen3.8-27B-Q4_K_M.gguf"

# The Q4_K_M file (17.1 GB) is larger than the T4's 16 GB, so only part of the model goes on the GPU.
# The remaining layers run on the CPU out of system RAM (~12 GB), which is much slower.
# Raise N_GPU_LAYERS until you hit an out-of-memory error, then back off a few layers.
N_GPU_LAYERS = 48     # the model has about 64 layers; 99 would offload all and fail on a T4
CTX_SIZE     = 16384  # agents (dsh) need a long context; lower it if you run out of VRAM
REBUILD_LLAMA = False   # True = ignore the cached build, pull the latest llama.cpp, rebuild, and re-save it

## 3. Get llama.cpp: restore the build from Google Drive (first run builds with CUDA for the T4, compute capability 7.5, and saves it)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")

BUILD_DIR = "/content/drive/MyDrive/llm_models/builds"
os.makedirs(BUILD_DIR, exist_ok=True)
TARBALL = f"{BUILD_DIR}/llama.cpp_cuda_sm75.tar"
SRC, BIN = "/content/llama.cpp", "/content/llama.cpp/build/bin"

def works():
    try:
        return subprocess.run([f"{BIN}/llama-server", "--version"], capture_output=True, timeout=120).returncode == 0
    except Exception:
        return False

if not REBUILD_LLAMA and os.path.exists(TARBALL):
    print("Restoring cached build from Drive...")
    os.makedirs(f"{SRC}/build", exist_ok=True)
    subprocess.run(["tar", "-xf", TARBALL, "-C", f"{SRC}/build"], check=True)

if works():
    print("Cached build works.")
else:
    print("No usable cached build; building (several minutes)...")
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        open("/content/llama_build.log", "a").write(r.stdout + r.stderr)
        if r.returncode:
            raise RuntimeError(f"Failed: {cmd}\n" + (r.stdout + r.stderr)[-3000:])
    open("/content/llama_build.log", "w").close()
    sh(f"rm -rf {SRC} && git clone --depth 1 https://github.com/ggml-org/llama.cpp {SRC}")
    sh(f"cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF")
    sh(f"cd {SRC} && cmake --build build --config Release -j $(nproc) --target llama-server llama-cli")
    rev = subprocess.run(f"cd {SRC} && git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
    open(f"{SRC}/build/bin/BUILD_INFO.txt", "w").write(f"llama.cpp {rev}\n")
    subprocess.run(["tar", "-cf", TARBALL + ".part", "-C", f"{SRC}/build", "bin"], check=True)
    os.rename(TARBALL + ".part", TARBALL)
    print("Saved build to Drive:", TARBALL)

In [ ]:
print(os.listdir(BIN)[:20])
assert os.path.exists(f"{BIN}/llama-server"), "Build failed; see /content/llama_build.log"
print(open(f"{BIN}/BUILD_INFO.txt").read() if os.path.exists(f"{BIN}/BUILD_INFO.txt") else "")

## 4. Get the model (from Google Drive if saved, otherwise download once and save)

In [ ]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/llm_models"   # permanent copy
# fast local disk; llama.cpp loads from here. The A100 80 GB adds a bigger NVMe disk at /mnt/local-scratch.
LOCAL_DIR = ("/mnt/local-scratch/models" if os.path.isdir("/mnt/local-scratch") and os.access("/mnt/local-scratch", os.W_OK)
             else "/content/models")
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(LOCAL_DIR, exist_ok=True)
drive_path = f"{DRIVE_DIR}/{FILENAME}"
MODEL_PATH = f"{LOCAL_DIR}/{FILENAME}"

if not os.path.exists(drive_path):
    %pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    print("Not on Drive yet; downloading from Hugging Face...")
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir=LOCAL_DIR)
    shutil.copyfile(MODEL_PATH, drive_path + ".part")   # .part so a cut-off copy is never mistaken for a full file
    os.rename(drive_path + ".part", drive_path)
    print("Saved to Drive:", drive_path)
elif not os.path.exists(MODEL_PATH):
    print("Copying from Drive to local disk (faster to load than reading from Drive)...")
    shutil.copyfile(drive_path, MODEL_PATH)
print(MODEL_PATH, f"{os.path.getsize(MODEL_PATH)/1e9:.1f} GB")
!df -h /content | tail -1

## 5. Start the server in the background (also serves a built-in chat web UI)

In [ ]:
import subprocess, time, requests, secrets

import os
KEY_FILE = "/content/drive/MyDrive/llm_models/hub/api_key"   # stable key on private Drive, shared with the hub notebooks
if os.path.exists(KEY_FILE) and len(open(KEY_FILE).read().strip()) >= 24:
    API_KEY = open(KEY_FILE).read().strip()
else:
    os.makedirs(os.path.dirname(KEY_FILE), exist_ok=True)
    API_KEY = secrets.token_urlsafe(32)
    open(KEY_FILE, "w").write(API_KEY + "\n")

server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server",
     "-m", MODEL_PATH, "-ngl", str(N_GPU_LAYERS), "-c", str(CTX_SIZE),
     "--host", "127.0.0.1", "--port", "8080", "--api-key", API_KEY, "--jinja"],
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])

## 6. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8080/v1/chat/completions", headers={"Authorization": f"Bearer {API_KEY}"}, json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": temperature,
    }, timeout=900)
    r.raise_for_status()
    m = r.json()["choices"][0]["message"]
    return m.get("reasoning_content") and ("[thinking]\n" + m["reasoning_content"] + "\n[answer]\n" + m["content"]) or m["content"]

print(chat("Explain what a GGUF file is in two sentences."))

## 7. Connect your machine over Tailscale

This joins the Colab VM to your Tailscale tailnet and publishes the server over HTTPS at `https://colab-llama-t4.<your-tailnet>.ts.net`, reachable only from devices in your tailnet. Run your harness (dsh or any OpenAI-compatible client) **on your own machine** in the same tailnet, pointed at the Base URL and API key printed below. Nothing harness-related runs in this notebook.

Create a **reusable + ephemeral** auth key at https://login.tailscale.com/admin/settings/keys. The first run asks for it and saves it to private Drive (`MyDrive/llm_models/hub/tailscale_authkey`); later sessions reuse it. The API key is also kept on Drive, and the hostname stays the same, so your harness settings keep working across sessions.

llama-server's own chat web UI is at the Base URL without `/v1`; it asks for the API key under Settings.

In [ ]:
# Uses the hub package's tunnel helpers (standard library only; nothing to install).
import os, sys, subprocess
HUB_DIR = "/content/hub_repo"
if not os.path.isdir(f"{HUB_DIR}/colab_llm_hub"):
    subprocess.run(f"rm -rf {HUB_DIR} && git clone -q --depth 1 https://github.com/Biegle-W/google_colab_llm {HUB_DIR}",
                   shell=True, check=True)
if HUB_DIR not in sys.path:
    sys.path.insert(0, HUB_DIR)
from colab_llm_hub import tunnel, provider

DRIVE_DIR   = "/content/drive/MyDrive/llm_models"
PORT        = 8080
TS_HOSTNAME = "colab-llama-t4"   # this server's name on your tailnet
TUNNEL      = "tailscale"  # @param ["tailscale", "cloudflare"]
# tailscale : private to your tailnet, stable URL (auth key saved on private Drive)
# cloudflare: no account; public https://<words>.trycloudflare.com URL, new every session (API key still applies)
TS_FUNNEL   = False   # True = also reachable from the public internet (the API key still applies)

if TUNNEL == "cloudflare":
    ts_proc, BASE = tunnel.start_cloudflare_quick(PORT, drive_dir=DRIVE_DIR, api_key=API_KEY)
else:
    ts_key = tunnel.load_or_ask_tailscale_authkey(f"{DRIVE_DIR}/hub/tailscale_authkey")
    ts_proc, BASE = tunnel.start_tailscale(PORT, ts_key, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
    del ts_key   # the key stays only in the private Drive file
MODEL_NAME = "local"  # llama-server answers to any model name
print("Web UI  :", BASE)
print("Base URL:", BASE + "/v1")
print("API key :", API_KEY)
print("Model   :", MODEL_NAME)
print()
print("On your machine" + (" (public URL, new each session)" if TUNNEL == "cloudflare" else " (same tailnet)")
      + ", any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print(f'curl {BASE}/v1/chat/completions -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" '
      f'-d \'{{"model": "{MODEL_NAME}", "messages": [{{"role": "user", "content": "Hello"}}]}}\'')

## 8. Cleanup and VRAM check

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
# server.terminate()   # stop the server when finished

## Notes
- Qwen3.8-27B is a hybrid thinking model, so replies may start with a reasoning block and use many tokens.
- The 17.1 GB Q4_K_M file does not fit in 16 GB of VRAM, so some layers run on the CPU. Expect a few tokens per second.
- The model is saved to `MyDrive/llm_models` on Google Drive, so later sessions copy it from Drive instead of downloading it again.
- The API key (`MyDrive/llm_models/hub/api_key`) and Tailscale auth key are kept on private Drive and the Tailscale hostname is fixed, so the Base URL and key stay the same across sessions. If the previous session's node is still listed in your tailnet, Tailscale adds a suffix (`-1`); remove the old node to keep the name.
- The compiled llama.cpp is cached on Drive (`MyDrive/llm_models/builds`). Set `REBUILD_LLAMA = True` to pick up a newer llama.cpp.
- If the build fails, read `/content/llama_build.log`.